In [1]:
import polars as pl
import numpy as np
import os
import io
import chess.pgn
import re
from typing import Dict, List, Any
from math import floor

In [2]:
INPUT = '../../../data/sample_data.pgn'

In [3]:
def parse_pgn_to_dataframe(pgn_text):
    """
    Parse a PGN (Portable Game Notation) file and convert it to a Polars DataFrame.
    
    Args:
        pgn_text (str): The contents of a PGN file as a string
        
    Returns:
        pl.DataFrame: A DataFrame containing parsed game information
    """
    games = []
    
    # Split the text into individual games
    game_blocks = re.split(r'\n\n(?=\[Event)', pgn_text.strip())
    
    for block in game_blocks:
        if not block.strip():
            continue
            
        game_data = {}
        
        # Extract header information
        headers = re.findall(r'\[(\w+)\s+"([^"]*)"\]', block)
        for key, value in headers:
            game_data[key] = value
        
        # Extract moves (everything after the last header until result)
        moves_match = re.search(r'\]\n\n(.+?)(?:\s+(?:1-0|0-1|1/2-1/2|\*))?$', block, re.DOTALL)
        if moves_match:
            moves_text = moves_match.group(1).strip()
            # Remove clock annotations and extra whitespace
            moves_clean = re.sub(r'\{[^}]*\}', '', moves_text)
            moves_clean = re.sub(r'\s+', ' ', moves_clean).strip()
            game_data['Moves'] = moves_clean
        else:
            game_data['Moves'] = ''
        
        games.append(game_data)
    
    # Create DataFrame
    if games:
        df = pl.DataFrame(games)
        
        # Convert numeric columns to appropriate types
        numeric_columns = ['WhiteElo', 'BlackElo', 'WhiteRatingDiff', 'BlackRatingDiff']
        for col in numeric_columns:
            if col in df.columns:
                df = df.with_columns(
                    pl.col(col).cast(pl.Int32, strict=False)
                )

        formats_to_keep = ["Rated Blitz game",
                       "Rated Bullet game",
                       "Rated Rapid game"]

        df = df.filter(pl.col('Event').str.contains('|'.join(formats_to_keep)))

        df = df.filter(pl.col('Termination') != 'Abandoned')

        df = df.with_columns(
        pl.when(pl.col("Event").str.contains("Rapid")).then(pl.lit("Rapid"))
        .when(pl.col("Event").str.contains("Bullet")).then(pl.lit("Bullet"))
        .when(pl.col("Event").str.contains("Blitz")).then(pl.lit("Blitz"))
        .otherwise(pl.col("Event"))
        .alias("TimeControl")
        )
    

        # calculate the number of moves
        def calculate_num_moves(moves: str) -> int:
            game = chess.pgn.read_game(io.StringIO(moves))
            if game is None:
                return 0
            num_moves = sum(1 for _ in game.mainline_moves()) // 2
            return num_moves
        
        df = df.with_columns(
            pl.col('Moves').map_elements(calculate_num_moves, return_dtype=pl.Int32).alias('NumMoves')
        )


        df = df.select(['TimeControl','WhiteElo', 'BlackElo',
                    'Termination', 'Moves','Result', 'NumMoves'])


        return df
    else:
        return pl.DataFrame()

# Read PGN file
with open(INPUT, 'r', encoding='utf-8') as file:
    pgn_text = file.read()
    df = parse_pgn_to_dataframe(pgn_text)

# 35, 27, 31

df.head()

TimeControl,WhiteElo,BlackElo,Termination,Moves,Result,NumMoves
str,i32,i32,str,str,str,i32
"""Bullet""",1706,1671,"""Time forfeit""","""1. d4 1... c5 2. e3 2... e6 3.…","""0-1""",35
"""Bullet""",2262,2191,"""Normal""","""1. d4 1... d6 2. Nf3 2... Nf6 …","""1-0""",26
"""Bullet""",2279,2339,"""Normal""","""1. d4 1... Nf6 2. Nf3 2... e6 …","""0-1""",31
"""Bullet""",971,1040,"""Normal""","""1. b4 1... e5 2. Bb2 2... f5 3…","""0-1""",25
"""Bullet""",1752,1737,"""Time forfeit""","""1. e4 1... e5 2. Nf3 2... d6 3…","""0-1""",41


In [ ]:
class FeatureEngineer:
    """Extract chess features from PGN moves at a specific move number."""
    
    PIECE_VALUES = {
        chess.PAWN: 1,
        chess.KNIGHT: 3,
        chess.BISHOP: 3,
        chess.ROOK: 5,
        chess.QUEEN: 9,
        chess.KING: 0
    }
    
    @staticmethod
    def get_board_at_move(moves: str, target_move: int = 15) -> chess.Board:
        """
        Get the board state at a specific move number.
        
        Args:
            moves: PGN moves string
            target_move: Move number to analyze (default: 15)
            
        Returns:
            chess.Board object at the target move, or None if move doesn't exist
        """
        game: chess.pgn.Game = chess.pgn.read_game(io.StringIO(moves))

        board = game.board()
        move_count = 0
        
        for move in game.mainline_moves():
            board.push(move)
            move_count += 1
            # Each pair of moves (white + black) = 1 full move
            if move_count == target_move * 2:
                return board

    @staticmethod
    def calculate_material(board: chess.Board) -> Dict[str, int]:
        """
        Calculate material count for both sides.
        
        Args:
            board: Chess board state
            
        Returns:
            Dictionary with white_material, black_material, and material_diff
        """
        white_material = 0
        black_material = 0
        
        for square in chess.SQUARES:
            piece = board.piece_at(square)
            if piece:
                value = ChessFeatureExtractor.PIECE_VALUES[piece.piece_type]
                if piece.color == chess.WHITE:
                    white_material += value
                else:
                    black_material += value
        
        return {
            'white_material': white_material,
            'black_material': black_material,
            'material_diff': white_material - black_material
        }
    